<a href="https://colab.research.google.com/github/deepakri201/SR_for_NLST_Sybil/blob/main/demo/NLST_Sybil_FM_demo_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLST_Sybil_FM_demo_part1

In this notebook, we download and parse the DICOM SR files, and join with the IDC metadata.

Deepa Krishnaswamy

Brigham and Women's Hospital

July 2025

Notes:
- Colab Pro

# Parameterization

In [1]:
#@title Enter your Project ID here
# initialize this variable with your Google Cloud Project ID!
project_name = "idc-external-018" #@param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = project_name

!gcloud config set project $project_name

from google.colab import auth
auth.authenticate_user()

Updated property [core/project].


# Environment Setup

In [2]:
!pip install highdicom
!pip install pydicom
!pip install idc-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 75.1 MB/s eta 0:00:00
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.3.2
    Uninstalling duckdb-1.3.2:
      Successfully uninstalled duckdb-1.3.2


In [3]:
import os
import sys
import time

import numpy as np
import pandas as pd
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

import json
from pathlib import Path

import pydicom
from pydicom.sr.codedict import codes

In [4]:
from google.cloud import bigquery
from google.cloud import storage

In [5]:
import highdicom as hd
hd.__version__

'0.26.0'

In [6]:
pydicom.__version__

'3.0.1'

In [7]:
from idc_index import IDCClient

idc_client = IDCClient.client()

# Download and parse the SRs that hold the Sybil annotations and get metadata

## Download the SRs

In [9]:
%%capture

sr_directory_bucket = "gs://sr_nlst_sybil"
sr_directory = "/content/sr"
if not os.path.isdir(sr_directory):
  os.mkdir(sr_directory)

!gsutil -m cp -r $sr_directory_bucket $sr_directory

## Parse the SRs and form df

### df_sr_info - Get the PatientID, SeriesInstanceUID and sr_filename

In [10]:
# Get the list of files and split by PatientID and SeriesInstanceUID
# Get the info df

sr_filenames = []
for root, _, files in os.walk(os.path.join(sr_directory, 'sr_nlst_sybil')):
  for file in files:
    full_file_path = os.path.join(root, file)
    sr_filenames.append(full_file_path)

PatientIDs = [f.split('/')[4] for f in sr_filenames]
SeriesInstanceUIDs = [Path(f.split('/')[5]).stem for f in sr_filenames]

df_sr_info = pd.DataFrame()
df_sr_info['PatientID'] = PatientIDs
df_sr_info['SeriesInstanceUID'] = SeriesInstanceUIDs
df_sr_info['sr_filename'] = sr_filenames

print(len(df_sr_info))
df_sr_info.head()

970


,PatientID,SeriesInstanceUID,sr_filename
0,122866,1.2.840.113654.2.55.80557134350152339289139946...,/content/sr/sr_nlst_sybil/122866/1.2.840.11365...
1,122866,1.2.840.113654.2.55.24304037083340386593566212...,/content/sr/sr_nlst_sybil/122866/1.2.840.11365...
2,126446,1.2.840.113654.2.55.10223655654547815709262155...,/content/sr/sr_nlst_sybil/126446/1.2.840.11365...
3,106046,1.2.840.113654.2.55.56880820416622279507487194...,/content/sr/sr_nlst_sybil/106046/1.2.840.11365...
4,108061,1.2.840.113654.2.55.10807774762936711010605398...,/content/sr/sr_nlst_sybil/108061/1.2.840.11365...


### df_sr - all the info extracted from the SRs

In [11]:
# Now load the SRs and extract the useful information
# StudyInstanceUID, referenced SeriesInstanceUID, SOPInstanceUID for each slice, polyline

num_files = len(sr_filenames)
print('num_files: ' + str(num_files))

df_sr = pd.DataFrame()

patient_id_list = []
referenced_series_instance_uid_list = []
tracking_identifier_list = []
tracking_uid_list = []
finding_type_list = []
finding_site_list = []
referenced_sop_instance_uid_list = []
width_list = []
height_list = []
center_x_list = []
center_y_list = []

for index, sr_filename in enumerate(sr_filenames,1):

  sr = hd.sr.srread(sr_filename)

  # get the PatientID
  patient_id = sr.PatientID

  # get the referenced SeriesInstanceUID
  referenced_series_instance_uid = sr.CurrentRequestedProcedureEvidenceSequence[0].ReferencedSeriesSequence[0].SeriesInstanceUID

  # get the image_region_code
  image_region_code = codes.DCM.ImageRegion

  # will store the info needed for table too.
  poly_infos = []

  # First get the planar roi measurement gorups
  groups = sr.content.get_planar_roi_measurement_groups()

  # For each group, get the tracking ids, finding type and site, referenced SOP, and bbox
  for group in groups:

    # Get the tracking ids
    tracking_identifier = group.tracking_identifier
    tracking_uid = group.tracking_uid

    # Get the findings and finding_sites
    finding_type = [group.finding_type.CodeValue, group.finding_type.CodingSchemeDesignator, group.finding_type.CodeMeaning]
    finding_sites = []
    for finding_site in group.finding_sites:
      finding_sites.append([finding_site.value.CodeValue,
                            finding_site.value.CodingSchemeDesignator,
                            finding_site.value.CodeMeaning])
    # Get the Image Region
    referenced_sop_instance_uid = group.roi.ContentSequence[0].referenced_sop_instance_uid
    bbox = group.roi.value

    # calculate the width, height and center, as these are needed for display
    min_x = np.min([bbox[0,0], bbox[1,0], bbox[2,0], bbox[3,0]]) # using roi.GraphicData: min_x = np.min([bbox[0], bbox[2], bbox[4], bbox[6]])
    max_x = np.max([bbox[0,0], bbox[1,0], bbox[2,0], bbox[3,0]]) # using roi.GraphicData: max_x = np.max([bbox[0], bbox[2], bbox[4], bbox[6]])
    min_y = np.min([bbox[0,1], bbox[1,1], bbox[2,1], bbox[3,1]]) # using roi.GraphicData: min_y = np.min([bbox[1], bbox[3], bbox[5], bbox[7]])
    max_y = np.max([bbox[0,1], bbox[1,1], bbox[2,1], bbox[3,1]]) # using roi.GraphicData: max_y = np.max([bbox[1], bbox[3], bbox[5], bbox[7]])
    width = max_x - min_x
    height = max_y - min_y
    center_x = min_x + width/2
    center_y = min_y + height/2

    # append
    patient_id_list.append(patient_id)
    referenced_series_instance_uid_list.append(referenced_series_instance_uid)
    tracking_identifier_list.append(tracking_identifier)
    tracking_uid_list.append(tracking_uid)
    referenced_sop_instance_uid_list.append(referenced_sop_instance_uid)
    finding_type_list.append(finding_type)
    finding_site_list.append(finding_sites)
    width_list.append(width)
    height_list.append(height)
    center_x_list.append(center_x) # check if negation needed, was needed for Slicer
    center_y_list.append(center_y) # check if negation needed, was needed for Slicer

  # Print for every 10% processed
  if (index + 1) % (num_files // 10) == 0:
    print(f"Processed {((index + 1) / num_files) * 100:.0f}% of files")

# Form dataframe
df_sr['PatientID'] = patient_id_list
df_sr['SeriesInstanceUID'] = referenced_series_instance_uid_list # referenced series instance uid - image, not the SR.
df_sr['TrackingIdentifier'] = tracking_identifier_list
df_sr['TrackingUID'] = tracking_uid_list
df_sr['SOPInstanceUID'] = referenced_sop_instance_uid_list
df_sr['FindingType'] = finding_type_list
df_sr['FindingSite'] = finding_site_list
df_sr['width'] = width_list
df_sr['height'] = height_list
df_sr['center_x'] = center_x_list
df_sr['center_y'] = center_y_list

num_files: 970
Processed 10% of files
Processed 20% of files
Processed 30% of files
Processed 40% of files
Processed 50% of files
Processed 60% of files
Processed 70% of files
Processed 80% of files
Processed 90% of files
Processed 100% of files


In [12]:
df_sr.head()

,PatientID,SeriesInstanceUID,TrackingIdentifier,TrackingUID,SOPInstanceUID,FindingType,FindingSite,width,height,center_x,center_y
0,122866,1.2.840.113654.2.55.80557134350152339289139946...,3,1.2.826.0.1.3680043.8.498.58837213884938860841...,1.2.840.113654.2.55.85140172863189651811817196...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",33.046875,28.828125,-95.625000,-126.625000
1,122866,1.2.840.113654.2.55.80557134350152339289139946...,2,1.2.826.0.1.3680043.8.498.54854323183661596059...,1.2.840.113654.2.55.28322194281293620277197472...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",33.046875,29.531250,-95.625000,-127.679688
2,122866,1.2.840.113654.2.55.80557134350152339289139946...,5,1.2.826.0.1.3680043.8.498.36619552277260220867...,1.2.840.113654.2.55.24155502295703095887689309...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",32.343750,26.718750,-95.976562,-126.273438
3,122866,1.2.840.113654.2.55.80557134350152339289139946...,4,1.2.826.0.1.3680043.8.498.57923085001197713872...,1.2.840.113654.2.55.23182863798442199631529157...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",33.046875,27.421875,-95.625000,-126.625000
4,122866,1.2.840.113654.2.55.80557134350152339289139946...,6,1.2.826.0.1.3680043.8.498.41490433162736288242...,1.2.840.113654.2.55.28942721755342122764631011...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",32.208389,26.092873,-96.388252,-125.897594


### df_idc - dim, pixel spacing, ipp from IDC

In [13]:
# From BigQuery:
#   Dimensions
#   Pixel spacing
#   IPP, especially IPP[2] for the z value of the bounding box

client_bq = bigquery.Client(project=project_name)

query = f"""
    SELECT
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      SOPInstanceUID,
      `Rows` as num_rows,
      `Columns` as num_columns,
      PixelSpacing,
      ImagePositionPatient
    FROM
      `bigquery-public-data.idc_current.dicom_all`
    WHERE
      SOPInstanceUID IN UNNEST(@referenced_sop_instance_uid_list)
    ORDER BY
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      ImagePositionPatient[SAFE_OFFSET(2)]
      """

job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("referenced_sop_instance_uid_list", "STRING", referenced_sop_instance_uid_list)])
df_idc = client_bq.query(query, job_config=job_config).to_dataframe()

In [14]:
# Reformat the PixelSpacing and the ImagePositionPatient columns
df_idc['pixel_spacing_x'] = [np.float32(f[0]) for f in df_idc['PixelSpacing'].values]
df_idc['pixel_spacing_y'] = [np.float32(f[1]) for f in df_idc['PixelSpacing'].values]
df_idc['ipp0'] = [np.float32(f[0]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp1'] = [np.float32(f[1]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp2'] = [np.float32(f[2]) for f in df_idc['ImagePositionPatient'].values]

df_idc = df_idc[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID',
                 'num_rows', 'num_columns',
                 'pixel_spacing_x', 'pixel_spacing_y',
                 'ipp0', 'ipp1', 'ipp2']]

In [15]:
df_idc.head()

,PatientID,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,num_rows,num_columns,pixel_spacing_x,pixel_spacing_y,ipp0,ipp1,ipp2
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29991037322734048580038819...,512,512,0.585938,0.585938,-149.707031,-319.707031,-76.400002
1,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.27443508115501826384206327...,512,512,0.585938,0.585938,-149.707031,-319.707031,-78.400002
2,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.16899951679153198601142574...,512,512,0.585938,0.585938,-149.707031,-319.707031,-80.400002
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.17481124987277919843449170...,512,512,0.585938,0.585938,-149.707031,-319.707031,-82.400002
4,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29574962348509387538142601...,512,512,0.585938,0.585938,-149.707031,-319.707031,-84.400002


### df_nlst_metadata - get the associated clinical metadata - for classification

In [16]:
# Rewrite the query below but faster
client = bigquery.Client(project=project_name, location='US') # since below can't mix US and us-central1
df_needed = df_idc[['PatientID', 'SeriesInstanceUID', 'SOPInstanceUID']].drop_duplicates()
table_id = "idc-external-018.nlst_sybil_fm_demo.needed_uids"
job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.job.WriteDisposition.WRITE_TRUNCATE
    )
client.load_table_from_dataframe(df_needed, table_id, job_config=job_config).result()

LoadJob<project=idc-external-018, location=US, id=fe11ab8e-5c3a-41e8-8278-8be8185f17b0>

In [17]:
# Here we get the staging data

query = """
WITH dicom_mapped AS (
  SELECT
    PatientID,
    StudyInstanceUID,
    StudyDate,
    SeriesInstanceUID,
    SOPInstanceUID,
    InstanceNumber,
    `Rows`,
    `Columns`,
    -- Mapping StudyDate to numerical values
    CASE StudyDate
      WHEN '1999-01-02' THEN 0
      WHEN '2000-01-02' THEN 1
      WHEN '2001-01-02' THEN 2
      ELSE 3
    END AS StudyDate_mapped,
    COUNT(*) OVER (PARTITION BY SeriesInstanceUID) AS sop_count_per_series
  FROM `bigquery-public-data.idc_current.dicom_all`
  WHERE SeriesInstanceUID IN (
    SELECT DISTINCT SeriesInstanceUID
    FROM `idc-external-018.nlst_sybil_fm_demo.needed_uids`
  )
)

SELECT
  ctab.dicom_patient_id AS PatientID,
  dicom_mapped.StudyInstanceUID,
  dicom_mapped.StudyDate,
  dicom_mapped.SeriesInstanceUID,
  dicom_mapped.SOPInstanceUID,
  ctab.sct_slice_num,
  ctab.study_yr,
  dicom_mapped.Rows,
  dicom_mapped.Columns,
  dicom_mapped.sop_count_per_series,
  -- Map de_stag to de_stag_mapped using your dictionary
  CASE prsn.de_stag
    WHEN '110' THEN 0
    WHEN '120' THEN 1
    WHEN '210' THEN 2
    WHEN '220' THEN 3
    WHEN '310' THEN 4
    WHEN '320' THEN 5
    WHEN '400' THEN 6
    ELSE -1
  END AS de_stag_mapped
FROM
  `bigquery-public-data.idc_current_clinical.nlst_ctab` AS ctab
JOIN
  `bigquery-public-data.idc_current_clinical.nlst_prsn` AS prsn
  ON prsn.dicom_patient_id = ctab.dicom_patient_id
JOIN
  dicom_mapped
  ON dicom_mapped.InstanceNumber = ctab.sct_slice_num
  AND ctab.study_yr = dicom_mapped.StudyDate_mapped
JOIN
  `idc-external-018.nlst_sybil_fm_demo.needed_uids` AS needed
  ON needed.PatientID = prsn.dicom_patient_id
  AND needed.SeriesInstanceUID = dicom_mapped.SeriesInstanceUID
  AND needed.SOPInstanceUID = dicom_mapped.SOPInstanceUID
WHERE
  -- Only keep rows where de_stag_mapped is not -1
  CASE prsn.de_stag
    WHEN '110' THEN 0 # "Stage IA"
    WHEN '120' THEN 1 # "Stage IB"
    WHEN '210' THEN 2 # "Stage IIA"
    WHEN '220' THEN 3 # "Stage IIB"
    WHEN '310' THEN 4 # "Stage IIIA"
    WHEN '320' THEN 5 # "Stage IIIB"
    WHEN '400' THEN 6 # "Stage IV"
    ELSE -1
  END != -1
"""
df_nlst_metadata = client.query(query).to_dataframe()

In [18]:
df_nlst_metadata.head()

,PatientID,StudyInstanceUID,StudyDate,SeriesInstanceUID,SOPInstanceUID,sct_slice_num,study_yr,Rows,Columns,sop_count_per_series,de_stag_mapped
0,110866,1.2.840.113654.2.55.42685297602370900191480074...,1999-01-02,1.2.840.113654.2.55.17220587240744170390288046...,1.2.840.113654.2.55.22416340924708356925857448...,58,0,512,512,140,0
1,103621,1.2.840.113654.2.55.30700507894515034571053584...,2000-01-02,1.2.840.113654.2.55.11418098206925124295595278...,1.2.840.113654.2.55.15555075296772631665653161...,44,1,512,512,187,3
2,133279,1.2.840.113654.2.55.43896830206945804905042744...,2001-01-02,1.2.840.113654.2.55.14574133286917559233323931...,1.2.840.113654.2.55.11234802340137254379189014...,29,2,512,512,173,0
3,110071,1.2.840.113654.2.55.26292429066750142196150834...,1999-01-02,1.2.840.113654.2.55.18748851615875144719707364...,1.2.840.113654.2.55.14191163830095264176970662...,47,0,512,512,160,1
4,112893,1.2.840.113654.2.55.24271591486575092392465054...,1999-01-02,1.2.840.113654.2.55.21993346375295563704048013...,1.2.840.113654.2.55.62539380954745487056247211...,45,0,512,512,147,0


## Join the tables to hold the SR info and clinical metadata info

In [19]:
# First join together the df_sr_info and df_sr - to add the sr_filename
df_sr_join = df_sr.merge(df_sr_info,
                         left_on=['SeriesInstanceUID'],
                         right_on=['SeriesInstanceUID'],
                         suffixes=('', '_right'))
# Drop the duplicate column from the right dataframe
df_sr_join = df_sr_join.drop(columns=['PatientID_right'])

# Then join with df_idc
df_sr_join = df_sr_join.merge(df_idc,
                              left_on=['SOPInstanceUID'],
                              right_on=['SOPInstanceUID'],
                              suffixes=('','_right'))
# Drop the duplicate column from the right dataframe
df_sr_join = df_sr_join.drop(columns=['PatientID_right','SeriesInstanceUID_right'])

# Then join with the df_nlst_metadata
df_sr_and_nlst = df_sr_join.merge(df_nlst_metadata,
                                  left_on=['SOPInstanceUID'],
                                  right_on=['SOPInstanceUID'],
                                  suffixes=('','_right'))
df_sr_and_nlst = df_sr_and_nlst.drop(columns=['PatientID_right','StudyInstanceUID_right','SeriesInstanceUID_right','num_rows', 'num_columns'])

# Reorder the columns
dr_sr_and_nlst = df_sr_and_nlst[['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr', 'SeriesInstanceUID', 'sr_filename', 'sop_count_per_series',
                                 'TrackingIdentifier', 'TrackingUID', 'SOPInstanceUID',
                                 'FindingType', 'FindingSite',
                                 'pixel_spacing_x', 'pixel_spacing_y',
                                 'width', 'height', 'center_x', 'center_y', 'ipp0', 'ipp1', 'ipp2',
                                 'sct_slice_num', 'de_stag_mapped']]
df_sr_and_nlst.head()



,PatientID,SeriesInstanceUID,TrackingIdentifier,TrackingUID,SOPInstanceUID,FindingType,FindingSite,width,height,center_x,...,ipp0,ipp1,ipp2,StudyDate,sct_slice_num,study_yr,Rows,Columns,sop_count_per_series,de_stag_mapped
0,122866,1.2.840.113654.2.55.80557134350152339289139946...,3,1.2.826.0.1.3680043.8.498.58837213884938860841...,1.2.840.113654.2.55.85140172863189651811817196...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",33.046875,28.828125,-95.625000,...,-179.648438,-345.648438,-177.600006,2000-01-02,34,1,512,512,152,0
1,126446,1.2.840.113654.2.55.10223655654547815709262155...,8,1.2.826.0.1.3680043.8.498.74654360360456174236...,1.2.840.113654.2.55.55197093585354713399712215...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",42.187500,39.843750,-108.593750,...,-200.000000,-200.000000,-252.630005,1999-01-02,104,0,512,512,143,6
2,126446,1.2.840.113654.2.55.10223655654547815709262155...,8,1.2.826.0.1.3680043.8.498.74654360360456174236...,1.2.840.113654.2.55.55197093585354713399712215...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",42.187500,39.843750,-108.593750,...,-200.000000,-200.000000,-252.630005,1999-01-02,104,0,512,512,143,6
3,108061,1.2.840.113654.2.55.10807774762936711010605398...,3,1.2.826.0.1.3680043.8.498.22582988325726371924...,1.2.840.113654.2.55.98174634521976004347766089...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",37.425781,33.515625,-40.632809,...,-148.720703,-312.720703,-134.100006,2000-01-02,59,1,512,512,152,0
4,108061,1.2.840.113654.2.55.17922076604756733648279336...,3,1.2.826.0.1.3680043.8.498.88771408274894460187...,1.2.840.113654.2.55.16614716909394342742616011...,"[52988006, SCT, Lesion]","[[39607008, SCT, Lung structure (body structur...",39.650393,36.099609,-44.976562,...,-151.204102,-325.204102,-172.300003,2001-01-02,81,2,512,512,168,0


In [20]:
# Let's get the counts of the de_stag_mapped
df_sr_and_nlst_counts = df_sr_and_nlst['de_stag_mapped'].value_counts().sort_index()
df_sr_and_nlst_counts

,count
de_stag_mapped,
0,424
1,115
2,21
3,19
4,64
5,48
6,87


# Temporarily save out csv file to Google Drive

In [23]:
df_sr_and_nlst.to_csv("/content/nlst_sybil_fm.csv")

In [21]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [28]:
!cp "/content/nlst_sybil_fm.csv" "/content/gdrive/MyDrive/Colab Notebooks/SR_NLST_Sybil/demo/"